# Modul 5 · Praktika: CNN ilə şəkil klassifikasiyası

Ev tapşırığında Fashion-MNIST üzərində işləmisiniz. Bu dəfə daha çətin dataset götürürük: **CIFAR-10**. Burada şəkillər rənglidir (32×32×3) və 10 sinifə bölünür: təyyarə, avtomobil, quş, pişik, maral, it, qurbağa, at, gəmi və yük maşını.

**Bu dərsdə nə edəcəyik:**

1. Isınma: konvolyusiyanı əl ilə yazacağıq.
2. CIFAR-10 datasetini yükləyib ona baxacağıq.
3. Adi neyron şəbəkəni (MLP) CNN ilə müqayisə edəcəyik.
4. Batch ölçüsünün öyrənməyə necə təsir etdiyini görəcəyik.
5. BatchNorm əlavə edib fərqi ölçəcəyik.
6. Gradient-in “partlamasını” görəcək və onu clipping ilə düzəldəcəyik.
7. CNN-in “nə gördüyünə” baxacağıq: filtrlər və feature map-lər.
8. Hazır ResNet-18 ilə transfer learning edəcəyik.
9. Mini yarış: kim ən dəqiq modeli quracaq?

**Necə işləməli:** `# TODO` yazılan yerləri öz kodunuzla doldurun. Hər bölmənin sonunda suallar var — cavabları qısa şəkildə yazın.

> 💡 Colab-da işləyirsinizsə, əvvəlcə GPU-nu yandırın: **Runtime → Change runtime type → T4 GPU**. GPU ilə bütün notebook təxminən 15–20 dəqiqəyə işləyir.

In [1]:
import time
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as T
from torch.utils.data import DataLoader, Subset, Dataset

torch.manual_seed(42)
np.random.seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Cihaz:", device)

Cihaz: cuda


## 1. konvolyusiyanı əl ilə yazırıq

Slayddakı nümunəni xatırlayın: 5×5 şəkil və 3×3 filtr. Filtr şəklin üzərində sürüşür, hər addımda xanaları vurub toplayırıq.

**Tapşırıq:** `my_conv2d` funksiyasını yazın. Padding yoxdur, stride = 1.

İpucu: iki `for` dövrü kifayətdir. Pəncərəni `image[i:i+k, j:j+k]` kimi kəsmək olar.

In [ ]:
image = np.array([[1, 1, 1, 0, 0],
                  [0, 1, 1, 1, 0],
                  [0, 0, 1, 1, 1],
                  [0, 0, 1, 1, 0],
                  [0, 1, 1, 0, 0]], dtype=np.float32)

kernel = np.array([[1, 0, 1],
                   [0, 1, 0],
                   [1, 0, 1]], dtype=np.float32)


def my_conv2d(image, kernel):
    H, W = image.shape
    k = kernel.shape[0]
    out = np.zeros((H - k + 1, W - k + 1), dtype=np.float32)
    # TODO: hər (i, j) üçün pəncərəni kəsin, filtrə vurun və cəmi out[i, j]-yə yazın
    for i in range(out.shape[0]):
        for j in range(out.shape[1]):
            window = image[i:i + k, j:j + k]
            out[i, j] = np.sum(window * kernel)
    ...
    return out


result = my_conv2d(image, kernel)
print(result)

İndi nəticəni PyTorch-un hazır funksiyası ilə yoxlayaq. Slayddakı kimi `[[4, 3, 4], [2, 4, 3], [2, 3, 4]]` alınmalıdır.

In [ ]:
torch_result = F.conv2d(torch.tensor(image)[None, None], torch.tensor(kernel)[None, None])[0, 0].numpy()
print(torch_result)
print("Düzdür! 🎉" if np.allclose(result, torch_result) else "Nəsə səhvdir, yenidən yoxlayın.")

## 2. CIFAR-10 datasetini yükləyirik

Şəkilləri tensora çeviririk və normallaşdırırıq: hər kanaldan ortalamanı çıxıb standart sapmaya bölürük. Bu, öyrənməni sabitləşdirir.

In [ ]:
mean = (0.4914, 0.4822, 0.4465)
std = (0.2470, 0.2435, 0.2616)
transform = T.Compose([T.ToTensor(), T.Normalize(mean, std)])

train_set = torchvision.datasets.CIFAR10("data", train=True, download=True, transform=transform)
test_set = torchvision.datasets.CIFAR10("data", train=False, download=True, transform=transform)
classes = train_set.classes

print("Train:", len(train_set), "şəkil")
print("Test: ", len(test_set), "şəkil")
print("Siniflər:", classes)

In [ ]:
def show_images(dataset, n=10):
    fig, axes = plt.subplots(1, n, figsize=(n * 1.6, 2))
    for ax, idx in zip(axes, np.random.choice(len(dataset), n, replace=False)):
        img, label = dataset[idx]
        img = img.permute(1, 2, 0).numpy() * np.array(std) + np.array(mean)  # normallaşdırmanı geri qaytarırıq
        ax.imshow(img.clip(0, 1))
        ax.set_title(classes[label], fontsize=9)
        ax.axis("off")
    plt.show()


show_images(train_set)

img, label = train_set[0]
print("Bir şəklin ölçüsü:", tuple(img.shape), "→ kanal × hündürlük × en")

Aşağıda öyrətmə və yoxlama üçün köməkçi funksiyalar var. Onları dəyişmək lazım deyil, sadəcə işə salın.

`train` funksiyası hər addımda gradient-in uzunluğunu (norm) da yadda saxlayır — bu, 6-cı bölmədə lazım olacaq.

In [ ]:
train_loader = DataLoader(train_set, batch_size=128, shuffle=True, num_workers=2)
test_loader = DataLoader(test_set, batch_size=512, num_workers=2)


def count_params(model):
    return sum(p.numel() for p in model.parameters())


def evaluate(model, loader):
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            correct += (model(x).argmax(1) == y).sum().item()
            total += y.size(0)
    return correct / total


def train(model, loader, epochs=5, lr=1e-3, optimizer_name="adam", clip=None, verbose=True):
    model.to(device)
    params = [p for p in model.parameters() if p.requires_grad]
    if optimizer_name == "adam":
        optimizer = torch.optim.Adam(params, lr=lr)
    else:
        optimizer = torch.optim.SGD(params, lr=lr, momentum=0.9)
    loss_fn = nn.CrossEntropyLoss()
    history = {"loss": [], "grad_norm": []}

    for epoch in range(epochs):
        model.train()
        start = time.time()
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            loss = loss_fn(model(x), y)
            optimizer.zero_grad()
            loss.backward()
            # clip=None olanda max_norm=inf: heç nə kəsilmir, sadəcə gradient-in uzunluğunu ölçürük
            norm = torch.nn.utils.clip_grad_norm_(params, max_norm=clip if clip is not None else float("inf"))
            optimizer.step()
            history["loss"].append(loss.item())
            history["grad_norm"].append(norm.item())
        if verbose:
            epoch_loss = np.mean(history["loss"][-len(loader):])
            print(f"Epoch {epoch + 1}/{epochs} · loss {epoch_loss:.3f} · {time.time() - start:.1f} san")
    return history


def smooth(values, k=20):
    values = np.array(values, dtype=float)
    if len(values) < k:
        return values
    return np.convolve(values, np.ones(k) / k, mode="valid")

## 3. Adi şəbəkə (MLP) və CNN: kim qalib gələcək?

Slaydı xatırlayın: adi şəbəkə şəkli uzun bir sıraya çevirir və qonşu piksellərin əlaqəsini görmür. Gəlin bunu praktikada yoxlayaq.

**Tapşırıq 3.1:** MLP qurun: `Flatten` → `Linear(3*32*32, 512)` → `ReLU` → `Linear(512, 256)` → `ReLU` → `Linear(256, 10)`.

In [ ]:
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        # TODO: self.net = nn.Sequential(...) — yuxarıdakı sxemə uyğun
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(3 * 32 * 32, 512), nn.ReLU(),
            nn.Linear(512, 256), nn.ReLU(),
            nn.Linear(256, 10),
        )
        ...

    def forward(self, x):
        return self.net(x)

**Tapşırıq 3.2:** CNN qurun. Üç blok olsun, hər blokda `Conv2d(kernel_size=3, padding=1)` → `ReLU` → `MaxPool2d(2)`:

| Blok | Giriş kanalları | Çıxış kanalları | Çıxış ölçüsü |
|---|---|---|---|
| 1 | 3 | 32 | 32 × 16 × 16 |
| 2 | 32 | 64 | 64 × 8 × 8 |
| 3 | 64 | 128 | 128 × 4 × 4 |

Sonra `classifier`: `Flatten` → `Linear(128*4*4, 256)` → `ReLU` → `Linear(256, 10)`.

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        # TODO: self.features = nn.Sequential(...) — 3 blok; self.classifier = nn.Sequential(...)
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),    # 32 × 16 × 16
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),   # 64 × 8 × 8
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),  # 128 × 4 × 4
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 256), nn.ReLU(),
            nn.Linear(256, 10),
        )
        ...

    def forward(self, x):
        return self.classifier(self.features(x))


# Ölçüləri yoxlayaq: features-dan sonra 128 × 4 × 4 alınmalıdır
x = torch.randn(1, 3, 32, 32)
print("features çıxışı:", tuple(SimpleCNN().features(x).shape))

İndi hər iki modeli eyni şəraitdə — 5 epoch — öyrədək və nəticələri müqayisə edək.

In [ ]:
mlp = MLP()
print("MLP öyrənir...")
train(mlp, train_loader, epochs=5)
mlp_acc = evaluate(mlp, test_loader)

cnn = SimpleCNN()
print("\nCNN öyrənir...")
train(cnn, train_loader, epochs=5)
cnn_acc = evaluate(cnn, test_loader)

print(f"\n{'Model':<10}{'Parametr':>14}{'Test dəqiqliyi':>18}")
print(f"{'MLP':<10}{count_params(mlp):>14,}{mlp_acc:>17.1%}")
print(f"{'CNN':<10}{count_params(cnn):>14,}{cnn_acc:>17.1%}")

**Suallar:**

1. Hansı modelin parametri çoxdur? Neçə dəfə?
2. Hansı model daha dəqiqdir?
3. Niyə daha az parametri olan model daha yaxşı nəticə göstərir?

## 4. Batch ölçüsü nəyə təsir edir?

Vaxta qənaət üçün 10 000 şəkillik kiçik bir hissə götürürük və hər modeli 2 epoch öyrədirik.

**Tapşırıq:** dövrün içində verilmiş `bs` ölçüsü ilə yeni `DataLoader` yaradın.

In [ ]:
small_train = Subset(train_set, range(10000))
batch_results = {}

for bs in [16, 128, 512]:
    # TODO: loader = DataLoader(small_train, batch_size=..., shuffle=True, num_workers=2)
    loader = DataLoader(small_train, batch_size=bs, shuffle=True, num_workers=2)
    ...
    model = SimpleCNN()
    start = time.time()
    hist = train(model, loader, epochs=2, verbose=False)
    batch_results[bs] = {
        "time": time.time() - start,
        "acc": evaluate(model, test_loader),
        "loss": hist["loss"],
    }
    print(f"batch_size={bs:<4} · vaxt {batch_results[bs]['time']:.1f} san · "
          f"iterasiya sayı {len(hist['loss'])} · test dəqiqliyi {batch_results[bs]['acc']:.1%}")

In [ ]:
plt.figure(figsize=(9, 4))
for bs, r in batch_results.items():
    steps = np.linspace(0, 2, len(r["loss"]))  # x oxu: epoch
    plt.plot(steps, r["loss"], alpha=0.6, label=f"batch_size = {bs}")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Batch ölçüsünə görə loss əyriləri")
plt.legend()
plt.show()

**Suallar:**

1. Hansı batch ölçüsü ilə 2 epoch daha tez bitdi? Niyə?
2. Hansı əyri daha çox “titrəyir”? Niyə?
3. Bizim halda hansı ölçü ən yaxşı balansı verdi?

## 5. BatchNorm əlavə edirik

**Tapşırıq:** `SimpleCNN`-i kopyalayın və hər `Conv2d`-dən sonra, `ReLU`-dan əvvəl `nn.BatchNorm2d(...)` əlavə edin. Mötərizədə həmin conv qatının çıxış kanallarının sayı yazılır.

Fərqi daha aydın görmək üçün hər iki modeli SGD ilə və nisbətən böyük learning rate (0.05) ilə öyrədəcəyik.

In [ ]:
class CNNWithBN(nn.Module):
    def __init__(self):
        super().__init__()
        # TODO: SimpleCNN-dəki features-ı yazın, amma hər Conv2d-dən sonra BatchNorm2d əlavə edin
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2),
        )
        ...
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 256), nn.ReLU(),
            nn.Linear(256, 10),
        )

    def forward(self, x):
        return self.classifier(self.features(x))


print("BatchNorm-suz:")
model_plain = SimpleCNN()
hist_plain = train(model_plain, train_loader, epochs=3, lr=0.05, optimizer_name="sgd")

print("\nBatchNorm ilə:")
model_bn = CNNWithBN()
hist_bn = train(model_bn, train_loader, epochs=3, lr=0.05, optimizer_name="sgd")

print(f"\nTest dəqiqliyi: BatchNorm-suz {evaluate(model_plain, test_loader):.1%}, "
      f"BatchNorm ilə {evaluate(model_bn, test_loader):.1%}")

In [ ]:
plt.figure(figsize=(9, 4))
plt.plot(smooth(hist_plain["loss"]), label="BatchNorm-suz")
plt.plot(smooth(hist_bn["loss"]), label="BatchNorm ilə")
plt.xlabel("İterasiya")
plt.ylabel("Loss (hamarlanmış)")
plt.title("BatchNorm öyrənməni necə dəyişir?")
plt.legend()
plt.show()

**Suallar:**

1. Hansı modelin loss-u daha tez düşür?
2. `evaluate` funksiyasında niyə `model.eval()` çağırırıq? BatchNorm olan modeldə bunu unutsaq nə olar?

## 6. Gradient “partlayanda”: clipping

İndi qəsdən pis şərait yaradırıq: BatchNorm-suz model və çox böyük learning rate (0.3). Gradient-in uzunluğunu izləyəcəyik.

Sonra eyni təcrübəni `clip=1.0` ilə təkrarlayacağıq — `train` funksiyası içində `clip_grad_norm_` artıq yazılıb.

> Nəticə hər işə salanda bir az fərqli ola bilər. Loss `nan` olursa, bu, gradient-in tam “partladığı” deməkdir.

In [ ]:
clip_loader = DataLoader(small_train, batch_size=128, shuffle=True, num_workers=2)

torch.manual_seed(0)
print("Clipping-siz:")
hist_noclip = train(SimpleCNN(), clip_loader, epochs=2, lr=0.3, optimizer_name="sgd")

torch.manual_seed(0)
print("\nClipping ilə (max_norm = 1.0):")
# TODO: eyni train(...) çağırışını yazın, amma clip=1.0 əlavə edin
hist_clip = train(SimpleCNN(), clip_loader, epochs=2, lr=0.3, optimizer_name="sgd", clip=1.0)
...

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(hist_noclip["grad_norm"], label="Clipping-siz", alpha=0.7)
axes[0].plot(hist_clip["grad_norm"], label="Clipping ilə", alpha=0.7)
axes[0].set_yscale("log")
axes[0].set_title("Gradient-in uzunluğu (kəsilməzdən əvvəl)")
axes[0].set_xlabel("İterasiya")
axes[0].legend()

axes[1].plot(hist_noclip["loss"], label="Clipping-siz", alpha=0.7)
axes[1].plot(hist_clip["loss"], label="Clipping ilə", alpha=0.7)
axes[1].set_title("Loss")
axes[1].set_xlabel("İterasiya")
axes[1].legend()
plt.show()

**Suallar:**

1. Clipping-siz halda loss və gradient-lə nə baş verdi?
2. Clipping hər problemi həll edirmi? `lr`-i 0.05 və 1.0 edib təcrübəni təkrarlayın.
3. Qrafikdə gradient clipping ilə də böyük görünür. Niyə? (İpucu: qrafikə kəsilməzdən **əvvəlki** uzunluq yazılır.)

## 7. CNN nə görür?

3-cü bölmədə öyrətdiyimiz `cnn` modelinə baxaq.

Əvvəlcə birinci qatın 32 filtrini şəkil kimi göstəririk. Hər filtr 3×3 ölçüdədir və 3 kanallıdır (RGB).

In [ ]:
filters = cnn.features[0].weight.detach().cpu()          # 32 × 3 × 3 × 3
filters = (filters - filters.min()) / (filters.max() - filters.min())  # 0–1 aralığına gətiririk

fig, axes = plt.subplots(4, 8, figsize=(10, 5))
for ax, f in zip(axes.flat, filters):
    ax.imshow(f.permute(1, 2, 0).numpy())
    ax.axis("off")
plt.suptitle("Birinci qatın filtrləri")
plt.show()

İndi bir şəkil götürüb birinci `Conv + ReLU` qatından sonra nə alındığına baxaq.

**Tapşırıq:** `feature_maps`-i hesablayın. İpucu: `cnn.features[:2]` — ilk Conv və ReLU qatlarıdır. Şəkil `img[None]` kimi batch formasına salınmalıdır.

In [ ]:
img, label = test_set[3]
cnn.eval()
with torch.no_grad():
    # TODO: feature_maps = cnn.features[:2](...)[0] — nəticə 32 × 32 × 32 olmalıdır
    feature_maps = cnn.features[:2](img[None].to(device))[0].cpu()
    ...

plt.figure(figsize=(2, 2))
plt.imshow((img.permute(1, 2, 0).numpy() * np.array(std) + np.array(mean)).clip(0, 1))
plt.title(f"Şəkil: {classes[label]}")
plt.axis("off")
plt.show()

fig, axes = plt.subplots(2, 8, figsize=(14, 3.8))
for i, ax in enumerate(axes.flat):
    ax.imshow(feature_maps[i], cmap="viridis")
    ax.set_title(f"#{i}", fontsize=9)
    ax.axis("off")
plt.suptitle("Birinci qatdan sonrakı 16 feature map")
plt.show()

**Sual:** feature map-lərdə nə görürsünüz? Bəziləri kənarlara, bəziləri isə müəyyən rənglərə reaksiya verir. Slayddakı “sadədən mürəkkəbə” sxemini xatırlayın.

## 8. Transfer learning: hazır ResNet-18

ResNet-18 ImageNet-də 1,2 milyon şəkil üzərində artıq öyrədilib. Biz onun “gözlərini” (feature extraction hissəsini) olduğu kimi saxlayacağıq və yalnız sonuncu qatı öz tapşırığımız üçün dəyişəcəyik.

Tapşırıq sadədir: **pişik və it** şəkillərini ayırmaq. Hər sinifdən cəmi 1000 şəkil götürürük.

> ResNet 224×224 ölçülü şəkillərlə öyrədildiyi üçün şəkilləri böyüdürük. Bu hissə GPU ilə təxminən 1–2 dəqiqə çəkir.

In [ ]:
from torchvision.models import resnet18, ResNet18_Weights

tl_transform = T.Compose([
    T.Resize(224),
    T.ToTensor(),
    T.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225)),  # ImageNet-in ortalamaları
])
CAT, DOG = classes.index("cat"), classes.index("dog")


class CatDog(Dataset):
    """CIFAR-10-dan yalnız pişik (0) və it (1) şəkillərini götürür."""

    def __init__(self, train, per_class):
        self.base = torchvision.datasets.CIFAR10("data", train=train, download=True, transform=tl_transform)
        targets = np.array(self.base.targets)
        cats = np.where(targets == CAT)[0][:per_class]
        dogs = np.where(targets == DOG)[0][:per_class]
        self.items = [(i, 0) for i in cats] + [(i, 1) for i in dogs]

    def __len__(self):
        return len(self.items)

    def __getitem__(self, n):
        idx, label = self.items[n]
        return self.base[idx][0], label


cd_train = DataLoader(CatDog(train=True, per_class=1000), batch_size=64, shuffle=True, num_workers=2)
cd_test = DataLoader(CatDog(train=False, per_class=500), batch_size=128, num_workers=2)
print("Train:", len(cd_train.dataset), "· Test:", len(cd_test.dataset))

**Tapşırıq:**

1. Bütün parametrləri “dondurun”: `requires_grad = False`.
2. Sonuncu qatı (`model_tl.fc`) 2 çıxışı olan yeni `nn.Linear` ilə əvəz edin. Giriş ölçüsünü `model_tl.fc.in_features`-dən götürün.

Yeni yaradılan qatın parametrləri avtomatik olaraq `requires_grad = True` olur, yəni yalnız o öyrənəcək.

In [ ]:
model_tl = resnet18(weights=ResNet18_Weights.DEFAULT)

# TODO: bütün parametrləri dondurun və model_tl.fc-ni nn.Linear(..., 2) ilə əvəz edin
for p in model_tl.parameters():
    p.requires_grad = False
model_tl.fc = nn.Linear(model_tl.fc.in_features, 2)
...

trainable = sum(p.numel() for p in model_tl.parameters() if p.requires_grad)
print(f"Cəmi parametr: {count_params(model_tl):,} · öyrənən: {trainable:,}")

train(model_tl, cd_train, epochs=2)
print(f"ResNet-18 (transfer learning): {evaluate(model_tl, cd_test):.1%}")

Müqayisə üçün eyni 2000 şəkil üzərində öz `SimpleCNN`-imizi sıfırdan öyrədək. Burada şəkillər 32×32 ölçüsündə qalır.

In [ ]:
class CatDog32(CatDog):
    def __init__(self, train, per_class):
        super().__init__(train, per_class)
        self.base.transform = transform  # 32×32 və CIFAR normallaşdırması


scratch = SimpleCNN()
scratch.classifier[-1] = nn.Linear(256, 2)
train(scratch, DataLoader(CatDog32(True, 1000), batch_size=64, shuffle=True, num_workers=2), epochs=5)
print(f"SimpleCNN (sıfırdan): {evaluate(scratch, DataLoader(CatDog32(False, 500), batch_size=128)):.1%}")

**Suallar:**

1. Hansı model daha yaxşı nəticə göstərdi? Fərq nə qədərdir?
2. ResNet-18-də 11 milyondan çox parametr var, amma biz cəmi bir neçə yüzünü öyrətdik. Bu necə işləyir?
3. Real həyatda transfer learning nə vaxt faydalıdır?

## 9. Mini yarış 🏆

**Qaydalar:**

- Dataset: CIFAR-10, tam train hissəsi.
- Ən çox **5 epoch**.
- Hazır (pretrained) modellər qadağandır — yalnız öz arxitekturanız.
- Hər şey olar: daha çox qat, BatchNorm, Dropout, data augmentation, başqa optimizer və learning rate.

İpucu: data augmentation üçün train transform-a `T.RandomHorizontalFlip()` və `T.RandomCrop(32, padding=4)` əlavə etməyi sınayın.

Sonda test dəqiqliyinizi lövhəyə yazın!

In [ ]:
class MyBestCNN(nn.Module):
    def __init__(self):
        super().__init__()
        # TODO: öz arxitekturanızı qurun
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(nn.Flatten(), nn.Linear(32 * 16 * 16, 10))

    def forward(self, x):
        return self.classifier(self.features(x))


my_model = MyBestCNN()
print(f"Parametr sayı: {count_params(my_model):,}")
train(my_model, train_loader, epochs=5)
print(f"\n🏆 Mənim nəticəm: {evaluate(my_model, test_loader):.1%}")

## Yekun

Bu gün praktikada gördük ki:

- CNN adi şəbəkədən **daha az parametrlə daha dəqiq** işləyir.
- Batch ölçüsü öyrənmənin **sürətinə və sabitliyinə** təsir edir.
- **BatchNorm** öyrənməni sürətləndirir və sabitləşdirir.
- **Gradient clipping** gradient partlayanda öyrənməni xilas edir, amma düzgün learning rate-i əvəz etmir.
- Birinci qatın filtrləri **kənarları və rəngləri** tapır.
- **Transfer learning** az data ilə belə yüksək nəticə verir.